# DermaLens · EfficientNet-B0 Kaggle training
Model 2/5. Train `real`, `synthetic`, or `combined` using **frozen train/val/test splits**. Saves best-validation checkpoints, predictions, preprocessing preview and result ZIP.

**Verification:** The notebook was authored using verified Torchvision APIs, but local Python execution was unavailable. Run the preflight cell and inspect previews before training. Official pretrained weights load on Kaggle.

In [ ]:
from pathlib import Path
REAL_SOURCE = "/kaggle/input/EDIT_REAL/facial_real_split.zip"
SYN_SOURCE = ""   # Add synthetic ZIP/folder to run synthetic or combined experiments
RUN_MODES = ["real"]   # ["real", "synthetic", "combined"]
PRETRAINED_WEIGHTS_PATH = ""  # Official efficientnet_b0_rwightman-7f5810bc.pth if offline
OUTPUT_ROOT = Path("/kaggle/working/dermalens_efficientnet_b0")
CLASSES = ["Acne", "Fungal_Infection", "Vitiligo", "Hyperpigmentation", "Skin_Cancer", "Normal"]
SEED = 42
SIZE = 224
BATCH_SIZE = 24
NUM_WORKERS = 2
EPOCHS_HEAD = 6
EPOCHS_PARTIAL = 15
EPOCHS_FULL = 0
PATIENCE = 5
LR_HEAD = 1e-3
LR_PARTIAL_HEAD = 1e-4
LR_PARTIAL_BACKBONE = 2e-5
LR_FULL_HEAD = 5e-5
LR_FULL_BACKBONE = 5e-6
WEIGHT_DECAY = 1e-4
RUN_PREFLIGHT = True


## Imports, source discovery and image verification
No additional pip installs are normally required for a current Kaggle PyTorch GPU notebook. If Torchvision is incompatible with installed PyTorch, use Kaggle's matching preinstalled versions rather than upgrading one independently.

In [ ]:
import os, random, json, csv, math, hashlib, time, shutil, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, classification_report

assert len(CLASSES) == len(set(CLASSES)) == 6
assert RUN_MODES and set(RUN_MODES) <= {"real", "synthetic", "combined"}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
LABEL_IDS = {name:i for i,name in enumerate(CLASSES)}
print("torch:",torch.__version__,"torchvision:",torchvision.__version__,"device:",DEVICE)
EXTS = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"}

def find_root(directory):
    d = Path(directory)
    if all((d/s).is_dir() for s in ("train","val","test")): return d
    hits = []
    for base, dirs, _ in os.walk(d):
        p=Path(base)
        if len(p.relative_to(d).parts)>4: dirs[:]=[]; continue
        if all((p/s).is_dir() for s in ("train","val","test")):
            hits.append(p); dirs[:]=[]
    if len(hits)!=1: raise ValueError(f"Found {len(hits)} split roots in {d}: {hits[:3]}")
    return hits[0]

def source_root(source, name):
    p=Path(source).expanduser()
    if not p.exists(): raise FileNotFoundError(f"{name} path missing: {p}")
    if p.is_dir(): return find_root(p)
    if p.suffix.lower()!=".zip": raise ValueError(f"Expected ZIP or folder: {p}")
    target=OUTPUT_ROOT/("_extracted_"+name)
    if target.exists(): shutil.rmtree(target)
    target.mkdir(parents=True)
    with zipfile.ZipFile(p) as z:
        for info in z.infolist():
            out=(target/info.filename).resolve()
            if not out.is_relative_to(target.resolve()):
                raise ValueError("Unsafe ZIP path")
        z.extractall(target)
    return find_root(target)

def load_rgb(path):
    with Image.open(path) as img:
        img.load()
        im=ImageOps.exif_transpose(img)
        if im.mode=="RGB": return im.copy()
        if "A" in im.getbands() or (im.mode=="P" and "transparency" in im.info):
            rgba=im.convert("RGBA")
            bg=Image.new("RGB",rgba.size,(255,255,255))
            bg.paste(rgba,mask=rgba.getchannel("A"))
            return bg
        return im.convert("RGB")

def inspect(path):
    try:
        im=load_rgb(path)
        if min(im.size)<64: return "too_small"
        a=np.asarray(im.resize((64,64),Image.Resampling.BILINEAR),dtype=np.float32).mean(2)
        if (a<8).mean()>.995 and a.std()<5: return "nearly_all_black"
        if (a>247).mean()>.995 and a.std()<5: return "nearly_all_white"
        return ""
    except (OSError,ValueError,UnidentifiedImageError,Image.DecompressionBombError) as e:
        return f"unreadable_{type(e).__name__}"

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as fh:
        for chunk in iter(lambda:fh.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

roots={}
roots["real"]=source_root(REAL_SOURCE,"real")
if SYN_SOURCE.strip(): roots["synthetic"]=source_root(SYN_SOURCE,"synthetic")
if any(m in RUN_MODES for m in ("synthetic","combined")) and "synthetic" not in roots:
    raise ValueError("Set SYN_SOURCE for synthetic/combined training")
records=[]; invalid=[]
for domain,root in roots.items():
    for split in ("train","val","test"):
        for cls in CLASSES:
            folder=root/split/cls
            if not folder.is_dir(): raise FileNotFoundError(str(folder))
            for p in sorted(folder.rglob("*")):
                if not p.is_file() or p.suffix.lower() not in EXTS: continue
                rec={"domain":domain,"split":split,"class_name":cls,"label":LABEL_IDS[cls],"path":str(p)}
                bad=inspect(p)
                if bad: invalid.append({**rec,"reason":bad})
                else: records.append({**rec,"sha256":sha256(p)})
if invalid:
    pd.DataFrame(invalid).to_csv(OUTPUT_ROOT/"invalid_images.csv",index=False)
    if any(x["split"]!="train" for x in invalid):
        raise ValueError(f"{len(invalid)} invalid image(s), including validation/test; see invalid_images.csv")
    print("Excluded invalid TRAIN images:",len(invalid))
if not records: raise ValueError("No valid images")
by_hash=defaultdict(list)
for r in records: by_hash[r["sha256"]].append(r)
leaks=[]
for h,group in by_hash.items():
    if len({x["split"] for x in group})>1 or len({x["class_name"] for x in group})>1:
        leaks.append({"sha256":h,"paths":"; ".join(x["path"] for x in group)})
if leaks:
    pd.DataFrame(leaks).to_csv(OUTPUT_ROOT/"leakage_conflicts.csv",index=False)
    raise ValueError("Exact duplicate leakage/conflicting labels detected. Resolve upstream.")
pd.DataFrame(records).to_csv(OUTPUT_ROOT/"training_input_manifest.csv",index=False)
for domain in roots:
    for split in ("train","val","test"):
        counts=Counter(r["class_name"] for r in records if r["domain"]==domain and r["split"]==split)
        if any(counts[c]==0 for c in CLASSES):
            raise ValueError(f"Missing class in {domain}/{split}: {dict(counts)}")
print(pd.DataFrame(records).groupby(["domain","split","class_name"]).size().unstack(fill_value=0))


## Model-specific preprocessing
EXIF-corrected RGB → aspect-preserving letterbox to 224 × 224 → tensor → ImageNet normalization. This **intentionally differs** from Torchvision EfficientNet-B0 official center-crop, to retain facial lesions near frame edges. Horizontal flips and mild rotations are **train-only**. We do not use vertical flips or extreme color jitter.

In [ ]:
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
PAD=(124,116,104)
class Letterbox:
    def __init__(self,size=224): self.size=size
    def __call__(self,img):
        fit=ImageOps.contain(img,(self.size,self.size),method=Image.Resampling.BICUBIC)
        canvas=Image.new("RGB",(self.size,self.size),PAD)
        canvas.paste(fit,((self.size-fit.width)//2,(self.size-fit.height)//2))
        return canvas
train_tf=transforms.Compose([Letterbox(SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7,interpolation=InterpolationMode.BILINEAR,fill=PAD),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([Letterbox(SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class FacialDataset(Dataset):
    def __init__(self,rows,transform): self.rows=list(rows); self.transform=transform
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows[i]
        x=self.transform(load_rgb(row["path"]))
        if x.shape!=(3,SIZE,SIZE) or not torch.isfinite(x).all():
            raise ValueError(f"Bad tensor for {row['path']}")
        return x,row["label"],row["path"]

def unnormalize(x):
    return (x.detach().cpu()*torch.tensor(STD)[:,None,None]+torch.tensor(MEAN)[:,None,None]).clamp(0,1).permute(1,2,0).numpy()

fig,axes=plt.subplots(2,6,figsize=(16,6))
for i,cls in enumerate(CLASSES):
    for j,domain in enumerate(("real","synthetic")):
        ax=axes[j,i]
        matches=[r for r in records if r["domain"]==domain and r["split"]=="train" and r["class_name"]==cls]
        if matches:
            arr=unnormalize(eval_tf(load_rgb(matches[0]["path"])))
            if not np.isfinite(arr).all() or arr.std()<0.007 or arr.mean()<0.025:
                raise ValueError(f"Blank/invalid preprocessing preview: {matches[0]['path']}")
            ax.imshow(arr)
        else: ax.text(.5,.5,"Not attached",ha="center",va="center")
        ax.set_title(domain+" / "+cls,fontsize=8); ax.axis("off")
fig.tight_layout(); fig.savefig(OUTPUT_ROOT/"preprocessing_preview.png",dpi=140)
plt.show(); plt.close(fig)
print("Preprocessing preview saved.")


## Model, fine-tuning phases and training utilities
The classification head has six outputs. BatchNorm running statistics remain frozen during training. Pretrained weights are required for research training, loaded through the supported Torchvision weights API or an attached official `.pth`.

In [ ]:
def subset(domain,split):
    return [r for r in records if r["domain"]==domain and r["split"]==split]

def worker_seed(worker_id):
    s=torch.initial_seed()%(2**32)
    random.seed(s); np.random.seed(s)

def loader(rows,train=False,mode="real"):
    if not rows: raise ValueError("Empty dataset")
    sampler=None
    if train and mode=="combined":
        cnt=Counter((r["domain"],r["class_name"]) for r in rows)
        weights=[1.0/cnt[(r["domain"],r["class_name"])] for r in rows]
        sampler=WeightedRandomSampler(weights,num_samples=len(rows),replacement=True,
                         generator=torch.Generator().manual_seed(SEED))
    return DataLoader(FacialDataset(rows,train_tf if train else eval_tf),
        batch_size=BATCH_SIZE,shuffle=(train and sampler is None),sampler=sampler,
        num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=(NUM_WORKERS>0),
        worker_init_fn=(worker_seed if NUM_WORKERS else None))

def new_model(pretrained=True):
    if pretrained and PRETRAINED_WEIGHTS_PATH:
        net=efficientnet_b0(weights=None)
        raw=torch.load(PRETRAINED_WEIGHTS_PATH,map_location="cpu",weights_only=True)
        net.load_state_dict(raw.get("state_dict",raw),strict=True)
        origin=PRETRAINED_WEIGHTS_PATH
    elif pretrained:
        try: net=efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        except Exception as exc:
            raise RuntimeError("Enable Kaggle Internet or attach the official EfficientNet-B0 ImageNet weights") from exc
        origin="EfficientNet_B0_Weights.IMAGENET1K_V1"
    else:
        net=efficientnet_b0(weights=None); origin="UNTRAINED_PREFLIGHT_ONLY"
    net.classifier[1]=nn.Linear(net.classifier[1].in_features,6)
    return net.to(DEVICE),origin

def freeze_bn(net):
    for m in net.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm):
            m.eval()
            for p in m.parameters(): p.requires_grad_(False)

def set_stage(net,stage):
    for p in net.parameters(): p.requires_grad_(False)
    if stage=="head":
        for p in net.classifier[1].parameters(): p.requires_grad_(True)
    elif stage=="partial":
        for p in net.features[-2:].parameters(): p.requires_grad_(True)
        for p in net.classifier[1].parameters(): p.requires_grad_(True)
    elif stage=="full":
        for p in net.parameters(): p.requires_grad_(True)
    else: raise ValueError(stage)
    freeze_bn(net)

def optimizer_for(net,stage):
    head=[p for p in net.classifier[1].parameters() if p.requires_grad]
    if stage=="head": groups=[{"params":head,"lr":LR_HEAD}]
    elif stage=="partial":
        bb=[p for p in net.features[-2:].parameters() if p.requires_grad]
        groups=[{"params":bb,"lr":LR_PARTIAL_BACKBONE},{"params":head,"lr":LR_PARTIAL_HEAD}]
    else:
        ids={id(p) for p in head}
        bb=[p for p in net.parameters() if p.requires_grad and id(p) not in ids]
        groups=[{"params":bb,"lr":LR_FULL_BACKBONE},{"params":head,"lr":LR_FULL_HEAD}]
    return torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)

def evaluate(net,dl,loss_fn=None):
    net.eval(); ys=[]; preds=[]; ps=[]; names=[]; losses=[]
    with torch.inference_mode():
        for x,y,paths in dl:
            x=x.to(DEVICE); y=y.to(DEVICE)
            with torch.amp.autocast("cuda",enabled=(DEVICE.type=="cuda")):
                logits=net(x)
                if loss_fn is not None: losses.append((float(loss_fn(logits,y)),len(y)))
            probs=logits.float().softmax(-1).cpu().numpy()
            ys+=y.cpu().tolist(); ps+=probs.tolist()
            preds+=probs.argmax(-1).tolist(); names+=list(paths)
    metrics=dict(accuracy=float(accuracy_score(ys,preds)),
        macro_f1=float(f1_score(ys,preds,labels=list(range(6)),average="macro",zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(ys,preds)),
        val_loss=(sum(v*n for v,n in losses)/sum(n for _,n in losses) if losses else None),
        n=len(ys))
    return metrics,ys,preds,ps,names

def better(a,b):
    return b is None or a["macro_f1"]>b["macro_f1"]+1e-6 or (
       abs(a["macro_f1"]-b["macro_f1"])<=1e-6 and a["val_loss"]<b["val_loss"]-1e-8)

def save_checkpoint(path,net,mode,stage,origin,metrics):
    torch.save({"architecture":"efficientnet_b0","model_state":{k:v.detach().cpu() for k,v in net.state_dict().items()},
        "classes":CLASSES,"mode":mode,"stage":stage,"origin":origin,
        "image_size":SIZE,"mean":MEAN,"std":STD,"preprocessing":"EXIF_RGB_letterbox_224",
        "seed":SEED,"best_validation":metrics},path)

def reload_checkpoint(path):
    ck=torch.load(path,map_location="cpu",weights_only=True)
    if ck["classes"]!=CLASSES or ck["architecture"]!="efficientnet_b0":
        raise ValueError("Checkpoint architecture/class mapping mismatch")
    net,_=new_model(pretrained=False)
    net.load_state_dict(ck["model_state"],strict=True)
    return net.eval(),ck


## Lightweight preflight (no pretrained download)
Runs only one image through preprocessing and the untrained architecture, checking shape, finite outputs, gradient flow, and checkpoint reload. This is **not** a substitute for full Kaggle training.

In [ ]:
if RUN_PREFLIGHT:
    net,_=new_model(pretrained=False)
    set_stage(net,"head")
    r=records[0]
    x,y,_=FacialDataset([r],eval_tf)[0]
    x=x[None].to(DEVICE)
    net.eval()
    logits=net(x)
    assert logits.shape==(1,6) and torch.isfinite(logits).all()
    loss=nn.CrossEntropyLoss()(logits,torch.tensor([y],device=DEVICE))
    loss.backward()
    assert net.classifier[1].weight.grad is not None
    tmp=OUTPUT_ROOT/"_preflight_checkpoint.pth"
    save_checkpoint(tmp,net,"preflight","head","UNTRAINED_PREFLIGHT_ONLY",
                    {"macro_f1":0.0,"val_loss":float(loss.item())})
    chk,meta=reload_checkpoint(tmp)
    with torch.inference_mode(): v=chk(x)
    assert v.shape==(1,6) and torch.isfinite(v).all()
    tmp.unlink()
    del net,chk
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("PASS: RGB preprocessing, EfficientNet-B0 forward/backward, checkpoint reload")


## Train and evaluate independently on real and synthetic tests
Checkpoint selection uses **validation only**. In combined mode, training uses both training partitions and selects using the real validation partition. The two frozen test sets are evaluated after training.

In [ ]:
def run_mode(mode):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if mode=="real":
        train_rows=subset("real","train"); val_rows=subset("real","val")
    elif mode=="synthetic":
        train_rows=subset("synthetic","train"); val_rows=subset("synthetic","val")
    else:
        train_rows=subset("real","train")+subset("synthetic","train")
        val_rows=subset("real","val")
    train_dl=loader(train_rows,train=True,mode=mode)
    val_dl=loader(val_rows)
    net,origin=new_model(pretrained=True)
    loss_fn=nn.CrossEntropyLoss()
    run_dir=OUTPUT_ROOT/("efficientnet_b0_"+mode); run_dir.mkdir(exist_ok=True)
    best_path=run_dir/("efficientnet_b0_"+mode+"_best.pth")
    head_path=run_dir/("efficientnet_b0_"+mode+"_head_only_best.pth")
    best=None; history=[]
    for stage,budget in (("head",EPOCHS_HEAD),("partial",EPOCHS_PARTIAL),("full",EPOCHS_FULL)):
        if not budget: continue
        if best is not None: net,_=reload_checkpoint(best_path)
        set_stage(net,stage)
        opt=optimizer_for(net,stage)
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,budget))
        scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda"))
        stage_best=None; stale=0
        for epoch in range(1,budget+1):
            net.train(); freeze_bn(net)
            running=0.; seen=0
            for x,y,_ in train_dl:
                x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda",enabled=(DEVICE.type=="cuda")):
                    out=net(x); loss=loss_fn(out,y)
                if not torch.isfinite(loss): raise ValueError("Non-finite loss")
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_((p for p in net.parameters() if p.requires_grad),1.0)
                scaler.step(opt); scaler.update()
                running+=float(loss.item())*len(y); seen+=len(y)
            score,*_=evaluate(net,val_dl,loss_fn)
            history.append({"mode":mode,"stage":stage,"epoch":epoch,
                "train_loss":running/seen,**{"validation_"+k:v for k,v in score.items()}})
            print(mode,stage,epoch,"val_macro_f1",round(score["macro_f1"],4))
            if better(score,stage_best):
                stage_best=score.copy(); stale=0
                if stage=="head": save_checkpoint(head_path,net,mode,stage,origin,score)
            else: stale+=1
            if better(score,best):
                best=score.copy(); save_checkpoint(best_path,net,mode,stage,origin,best)
            scheduler.step()
            if stale>=PATIENCE: break
        pd.DataFrame(history).to_csv(run_dir/"history.csv",index=False)
    if not best_path.exists(): raise ValueError("No best checkpoint produced")
    best_net,ck=reload_checkpoint(best_path)
    test_scores=[]
    for domain in roots:
        metrics,ys,preds,probs,paths=evaluate(best_net,loader(subset(domain,"test")))
        metrics.update({"train_mode":mode,"test_domain":domain,"best_stage":ck["stage"]})
        test_scores.append(metrics)
        frame=pd.DataFrame({"path":paths,"true_label":ys,"pred_label":preds,
              "true_class":[CLASSES[x] for x in ys],"pred_class":[CLASSES[x] for x in preds]})
        for k,c in enumerate(CLASSES): frame["prob_"+c]=[p[k] for p in probs]
        frame.to_csv(run_dir/("predictions_"+domain+".csv"),index=False)
        pd.DataFrame(confusion_matrix(ys,preds,labels=range(6)),index=CLASSES,columns=CLASSES).to_csv(run_dir/("confusion_"+domain+".csv"))
        report=classification_report(ys,preds,labels=range(6),target_names=CLASSES,zero_division=0,output_dict=True)
        (run_dir/("report_"+domain+".json")).write_text(json.dumps(report,indent=2))
    pd.DataFrame(test_scores).to_csv(run_dir/"test_metrics.csv",index=False)
    return test_scores

scores=[]
for mode in RUN_MODES: scores.extend(run_mode(mode))
pd.DataFrame(scores).to_csv(OUTPUT_ROOT/"all_test_metrics.csv",index=False)
print(pd.DataFrame(scores).to_string(index=False))


## Export result ZIP with verification
Kaggle **Output**: `efficientnet_b0_training_results.zip` holds checkpoints, image preview, metrics and reports; it excludes extracted input images.

In [ ]:
package=Path("/kaggle/working/efficientnet_b0_training_results.zip")
with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
    for f in sorted(OUTPUT_ROOT.rglob("*")):
        if not f.is_file(): continue
        rel=f.relative_to(OUTPUT_ROOT)
        if any(part.startswith("_extracted_") for part in rel.parts): continue
        archive.write(f,arcname=rel.as_posix())
with zipfile.ZipFile(package) as z:
    if z.testzip() is not None: raise RuntimeError("ZIP integrity check failed")
    mains=[x for x in z.namelist() if x.endswith("_best.pth") and "_head_only_" not in x]
    if len(mains)!=len(RUN_MODES): raise RuntimeError(f"Expected {len(RUN_MODES)} main checkpoints, got {len(mains)}")
print("DONE:",package,"bytes:",package.stat().st_size)
